In [4]:
from pathlib import Path

import numpy as np
import pandas as pd


DATA_PATH = Path("data/Train.csv")
OUTPUT_PATH = Path("data/sun_angle_latitudes_sample_wide.csv")
SELECTED_MEASUREMENTS = ["L3_AER_AI", "L3_NO2", "L3_O3"]

columns = pd.read_csv(DATA_PATH, nrows=0).columns
sun_angle_cols = [
    col
    for col in columns
    if col.startswith("L3_") and "solar_" in col and col.endswith("_angle")
]

sat_df = pd.read_csv(
    DATA_PATH,
    usecols=["Place_ID", "Date", *sun_angle_cols],
    parse_dates=["Date"],
)

sat_sample_df = sat_df.groupby("Place_ID", sort=False).head(5).copy()


def solar_declination_deg(dates):
    """Approximate solar declination angle from date using the NOAA Fourier series."""
    day_of_year = dates.dt.dayofyear.to_numpy(dtype=float)
    gamma = 2 * np.pi * (day_of_year - 1) / 365.0

    declination_rad = (
        0.006918
        - 0.399912 * np.cos(gamma)
        + 0.070257 * np.sin(gamma)
        - 0.006758 * np.cos(2 * gamma)
        + 0.000907 * np.sin(2 * gamma)
        - 0.002697 * np.cos(3 * gamma)
        + 0.00148 * np.sin(3 * gamma)
    )
    return np.degrees(declination_rad)


def calculate_latitude_candidates(zenith_deg, azimuth_deg, declination_deg):
    zenith_values = zenith_deg.to_numpy(dtype=float)
    azimuth_values = azimuth_deg.to_numpy(dtype=float)

    z = np.radians(zenith_values)
    A = np.radians(azimuth_values)
    delta = np.radians(declination_deg)

    sin_z = np.sin(z)
    cos_z = np.cos(z)
    sin_A = np.sin(A)
    cos_A = np.cos(A)
    sin_delta = np.sin(delta)

    radicand = 1 - (sin_z**2 * sin_A**2)
    R = np.sqrt(np.clip(radicand, 0.0, None))
    alpha = np.arctan2(sin_z * cos_A, cos_z)

    missing_data = (
        np.isnan(zenith_values)
        | np.isnan(azimuth_values)
        | ((zenith_values == 0) & (azimuth_values == 0))
    )
    singular_geometry = R <= 1e-12

    with np.errstate(divide="ignore", invalid="ignore"):
        asin_ratio = sin_delta / R
        no_real_solution = singular_geometry | (asin_ratio < -1.0000001) | (asin_ratio > 1.0000001)
        asin_term = np.arcsin(np.clip(asin_ratio, -1.0, 1.0))

    phi1_raw = np.degrees(asin_term - alpha)
    phi2_raw = np.degrees(np.pi - asin_term - alpha)

    invalid = missing_data | no_real_solution
    phi1_raw[invalid] = np.nan
    phi2_raw[invalid] = np.nan

    phi1 = np.where((phi1_raw >= -90) & (phi1_raw <= 90), phi1_raw, np.nan)
    phi2 = np.where((phi2_raw >= -90) & (phi2_raw <= 90), phi2_raw, np.nan)

    status1 = np.full(phi1.shape, "ok", dtype=object)
    status2 = np.full(phi2.shape, "ok", dtype=object)

    status1[np.isnan(phi1) & ~np.isnan(phi1_raw)] = "out_of_bounds"
    status2[np.isnan(phi2) & ~np.isnan(phi2_raw)] = "out_of_bounds"
    status1[no_real_solution & ~missing_data] = "no_real_solution"
    status2[no_real_solution & ~missing_data] = "no_real_solution"
    status1[missing_data] = "missing_data"
    status2[missing_data] = "missing_data"

    return phi1_raw, phi2_raw, phi1, phi2, status1, status2


declination = solar_declination_deg(sat_sample_df["Date"])
measurements = sorted({col.split("_solar_")[0] for col in sun_angle_cols})

latitude_parts = []
for measurement in measurements:
    azimuth_col = f"{measurement}_solar_azimuth_angle"
    zenith_col = f"{measurement}_solar_zenith_angle"

    if azimuth_col not in sat_df.columns or zenith_col not in sat_df.columns:
        continue

    (
        latitude_1_raw,
        latitude_2_raw,
        latitude_1,
        latitude_2,
        latitude_1_status,
        latitude_2_status,
    ) = calculate_latitude_candidates(
        zenith_deg=sat_sample_df[zenith_col],
        azimuth_deg=sat_sample_df[azimuth_col],
        declination_deg=declination,
    )

    latitude_parts.append(
        pd.DataFrame(
            {
                "row_id": sat_sample_df.index,
                "Place_ID": sat_sample_df["Place_ID"].to_numpy(),
                "Date": sat_sample_df["Date"].to_numpy(),
                "measurement": measurement,
                "solar_zenith_angle": sat_sample_df[zenith_col].to_numpy(),
                "solar_azimuth_angle": sat_sample_df[azimuth_col].to_numpy(),
                "latitude_1_raw": latitude_1_raw,
                "latitude_2_raw": latitude_2_raw,
                "latitude_1": latitude_1,
                "latitude_2": latitude_2,
                "latitude_1_status": latitude_1_status,
                "latitude_2_status": latitude_2_status,
            }
        )
    )

latitude_df = pd.concat(latitude_parts, ignore_index=True)
latitude_df["has_valid_latitude"] = latitude_df[["latitude_1", "latitude_2"]].notna().any(axis=1)
latitude_df["has_africa_range_candidate"] = latitude_df[["latitude_1", "latitude_2"]].abs().le(35).any(axis=1)

latitude_issue_summary = latitude_df.groupby("measurement").agg(
    rows=("row_id", "size"),
    no_valid_latitude=("has_valid_latitude", lambda value: (~value).sum()),
    no_africa_range_candidate=("has_africa_range_candidate", lambda value: (~value).sum()),
)
latitude_status_summary = (
    latitude_df.melt(
        id_vars="measurement",
        value_vars=["latitude_1_status", "latitude_2_status"],
        value_name="status",
    )
    .groupby(["measurement", "variable", "status"])
    .size()
    .unstack(fill_value=0)
)

selected_measurements = [measurement for measurement in SELECTED_MEASUREMENTS if measurement in measurements]

latitude_wide_df = latitude_df[
    latitude_df["measurement"].isin(selected_measurements)
].pivot(
    index=["row_id", "Place_ID", "Date"],
    columns="measurement",
    values=[
        "solar_zenith_angle",
        "solar_azimuth_angle",
        "latitude_1",
        "latitude_2",
        "latitude_1_raw",
        "latitude_2_raw",
        "latitude_1_status",
        "latitude_2_status",
    ],
)
latitude_wide_df.columns = [
    f"{measurement}_{value_name}"
    for value_name, measurement in latitude_wide_df.columns
]
latitude_wide_df = latitude_wide_df.reset_index()

latitude_wide_df.to_csv(OUTPUT_PATH, index=False)
latitude_wide_df.head()


,row_id,Place_ID,Date,measurement,solar_zenith_angle,solar_azimuth_angle,latitude_1_raw,latitude_2_raw,latitude_1,latitude_2,has_valid_latitude,has_africa_range_candidate
0,0,010Q650,2020-01-02,L3_AER_AI,22.358167,-61.736719,-35.499647,193.456830,-35.499647,NaN,True,False
1,1,010Q650,2020-01-03,L3_AER_AI,28.614804,-67.693509,-37.415570,194.017706,-37.415570,NaN,True,False
2,2,010Q650,2020-01-04,L3_AER_AI,34.296977,-78.342701,-35.534067,199.840082,-35.534067,NaN,True,False
3,3,010Q650,2020-01-05,L3_AER_AI,30.545393,-73.896572,-35.535558,196.943980,-35.535558,NaN,True,False
4,4,010Q650,2020-01-06,L3_AER_AI,26.899694,-68.612480,-35.536618,194.573381,-35.536618,NaN,True,False
5,94,05EC30X,2020-01-02,L3_AER_AI,17.462130,-73.607722,-29.130901,198.984506,-29.130901,NaN,True,True
6,95,05EC30X,2020-01-03,L3_AER_AI,24.265529,-77.285470,-30.792523,199.460214,-30.792523,NaN,True,True
7,96,05EC30X,2020-01-04,L3_AER_AI,30.779975,-86.016962,-29.153135,204.414697,-29.153135,NaN,True,True
8,97,05EC30X,2020-01-05,L3_AER_AI,26.629273,-82.714494,-29.153681,201.877216,-29.153681,NaN,True,True
9,98,05EC30X,2020-01-06,L3_AER_AI,22.529734,-78.636239,-29.155991,199.810585,-29.155991,NaN,True,True
